In [1]:
from IPython.display import HTML
with open("custom/style.css", "r") as f:
    custom_style = f.read()
hide_cell_style = "<style>.jp-Cell:has(.hide-me) { display: yes; }</style>"
HTML(f"<div class='hide-me'></div><style>{custom_style}</style>{hide_cell_style}")

<h1><center><font color='blue'>Introduction to Parallel Programming with MPI<br><br>Nonblocking Collectives</font></center></h1>

# <font color='blue'>Nonblocking Collectives</font>

## Nonblocking Collectives in MPI

- Similar to blocking collectives: nonblocking collective calls <font color='green'>including all ranks</font> of a communicator <br>
  - <font color='green'>All ranks must call the function!</font>
- <font color='green'>Nonblocking</font> variants (since MPI 3.0):buffer can be used after completion (`MPI_Wait*`/`MPI_Test*`)
- Local: not synchronization
- Multiple outstanding collective communications on same communicator
- Cannot interfere with point-to-point communication
  - Completely separate modes of operation!
- <font color='green'>Cannot interfere with blocking collective communication</font>
  - Such interference was allowed in point-to-point communication

## Collectives in MPI

- <font color='green'>Rules</font> for all collectives
  - Data type matching
  - No tags
  - Count must be exact, i.e., there is only one message length, buffer must be large enough
- Types:
  - <font color='green'>Synchronization</font> (barrier)
  - <font color='green'>Data movement</font> (broadcast, scatter, gather, all to all)
  - Collective <font color='green'>computation</font> (reduction, scan)
  - <font color='green'>Combinations</font> of data movement and computation (reduction + broadcast)
- General assumption: <font color='green'>MPI does a better job</font> at collectives <font color='green'>than you</font> trying to emulate them with point-to-point calls

## Nonblocking Barrier

- Nonblocking synchronization
```cpp
int MPI_Ibarrier(MPI_Comm comm, MPI_Request *request)
```
- Must be followed by an `MPI_Wait`
- Calling process enters the barrier, no synchronization happens
- Synchronization may happen asynchronously
- Overlapping synchronization with work: reducing idle time
- Comparison:
  1. `MPI_Ibarrier` $\Longrightarrow$ Work $\Longrightarrow$ `MPI_Wait`
  2. $\hspace{6.20cm}$ Work $\Longrightarrow$ `MPI_Barrier`
  - idle times before and after Work differ on each process and their sum!

## Collectives: Blocking vs. Nonblocking

- Broadcast
```cpp
int MPI_Bcast(void *buffer, int count, MPI_Datatype datatype, int root, MPI_Comm comm)
````
```cpp
int MPI_Ibcast(void *buffer, int count, MPI_Datatype datatype, int root, MPI_Comm comm, MPI_Request *request)
```
- Scatter
```cpp
int MPI_Scatter(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
    void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm)
int MPI_Iscatter(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
    void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm, MPI_Request *request)
```
- Gather
```cpp
int MPI_Gather(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
    void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm)
```
```cpp
int MPI_Igather(const void *sendbuf, int sendcount, MPI_Datatype sendtype,
    void *recvbuf, int recvcount, MPI_Datatype recvtype, int root, MPI_Comm comm, MPI_Request *request)
```

<div class="callout note">
<strong>Note:</strong>
<span style="color:darkblue">
All nonblocking variants have an extra argument of <code>MPI_Request</code></span>
</div>

- Similarly many blocking collective calls have nonblocking analogues:
  - `MPI_Iallgather`, `MPI_Ialltoall`, `MPI_Ireduce`, ...

Remarks for MPI_Ireduce:
- Both <font color='green'>send</font> and <font color='green'>receive buffers</font> can be used only after completion!
- Similar to nonblocking point-to-point calls, `MPI_Wait*` or `MPI_Test*` must be used to halt or examine for the completion of a request, respectively
- `root` (if available) and `comm` must be the same on all processes
- <font color='green'>Type signature</font> of send and receive variables must match

## Nonblocking reduction on all ranks

```cpp
int MPI_Iallreduce(const void *sendbuf, void *recvbuf, int count,
                   MPI_Datatype datatype, MPI_Op op, MPI_Comm comm,
                   MPI_Request *request)
```
- No `root`
- `sendbuf` and `recvbuf` can be reused after completion: requires `MPI_Wait` or `MPI_Test`
- `recvbuf` is <font color='green'>significant on all processes</font>



## Quiz:

1. Nonblocking collectives are useful when there exist multiple collective calls on the same communicator so <font color='green'>overlapping different collective calls</font> becomes possible.
   <ol style="list-style-type: lower-alpha;">
   <li>Correct</li>
   <li>Incorrect</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> a.
   </details> <p></p>
  
2. Nonblocking collectives can <font color='green'>interfere</font> with blocking collectives, that is, some processes of the communicator call the nonblocking binding and the rest blocking counterpart.
   <ol style="list-style-type: lower-alpha;">
   <li>Correct</li>
   <li>Incorrect</li>
   </ol>
   <details style="display: inline;">
       <summary style="display: inline; cursor: pointer;">
           <b>Answer: </b>
       </summary> b.
   </details> <p></p>